### Homework 5: Question search engine

Remeber week01 where you used GloVe embeddings to find related questions? That was.. cute, but far from state of the art. It's time to really solve this task using context-aware embeddings.

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [ ]:
%pip install --upgrade transformers datasets accelerate deepspeed sentencepiece protobuf pandas
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets
import numpy as np

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 32.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 122.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 21.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 129.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 49.9 MB/s  0:00:01
  Created wheel for deepspeed: filename=deepspeed-0.19.9-py3-none-any.whl size=2192418 sha256=c1be89afb23bb4a81400c9b5e24c5d20196608fd660e9c7297bd00775108bc4a
  Stored in directory: /root/.cache/pip/wheels/99/42/13/14d3508ae75d66e803dff6d82342500415196432f3e10cafe1
Successfully built deepspeed
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: pro

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


### Load data and model

In [2]:
qqp = datasets.load_dataset('SetFit/qqp')
print('\n')
print("Sample[0]:", qqp['train'][0])
print("Sample[3]:", qqp['train'][3])

README.md:   0%|          | 0.00/313 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl: reconstructing file:   0%|          |  0.00B / 70.8MB            

train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/7.83M [00:00<?, ?B/s]

test.jsonl: reconstructing file:   0%|          |  0.00B / 76.0MB            

test.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]



Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [3]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  433MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### Tokenize the data

In [4]:
MAX_LENGTH = 128
def preprocess_function(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [5]:
print(repr(qqp_preprocessed['train'][0]['input_ids'])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Task 1: evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [6]:
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [7]:
for batch in val_loader:
     break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch['input_ids'],
        attention_mask=batch['attention_mask'],
        token_type_ids=batch['token_type_ids']
    )

print('\nPrediction (probs):', torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

__Your task__ is to measure the validation accuracy of your model.
Doing so naively may take several hours. Please make sure you use the following optimizations:

- run the model on GPU with no_grad
- using batch size larger than 1
- use optimize data loader with num_workers > 1
- (optional) use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [8]:
import time
import gc

assert torch.cuda.is_available(), "Please enable a GPU runtime in Colab."
device = torch.device('cuda')
eval_batch_size = 32
num_workers = 2
print('GPU:', torch.cuda.get_device_name(device))
print('PyTorch:', torch.__version__, '| Transformers:', transformers.__version__)

def evaluate_model(current_model, encoded_data, current_tokenizer):
    input_names = [
        name for name in current_tokenizer.model_input_names
        if name in encoded_data.column_names
    ]
    data = encoded_data.select_columns(input_names + ['label']).with_format('torch')
    loader = torch.utils.data.DataLoader(
        data, batch_size=eval_batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True, persistent_workers=True,
        collate_fn=transformers.default_data_collator
    )
    current_model = current_model.to(device).float()
    current_model.eval()

    with torch.no_grad():
        # warmed up the GPU before timing
        for step, batch in enumerate(loader):
            inputs = {name: batch[name].to(device, non_blocking=True) for name in input_names}
            current_model(**inputs)
            if step == 4:
                break

        correct = torch.zeros((), dtype=torch.long, device=device)
        torch.cuda.synchronize(device)
        start = time.perf_counter()
        for batch in loader:
            inputs = {name: batch[name].to(device, non_blocking=True) for name in input_names}
            labels = batch['labels'].to(device, non_blocking=True)
            logits = current_model(**inputs).logits
            correct += (logits.argmax(dim=1) == labels).sum()
        torch.cuda.synchronize(device)
        elapsed = time.perf_counter() - start

    # counted tensor storage, not download size or peak GPU memory
    size_bytes = sum(p.numel() * p.element_size() for p in current_model.parameters())
    size_bytes += sum(b.numel() * b.element_size() for b in current_model.buffers())
    return {
        'accuracy': correct.item() / len(data),
        'samples_per_second': len(data) / elapsed,
        'size_MB': size_bytes / 1_000_000
    }

baseline_result = {'model': model_name, **evaluate_model(model, val_set, tokenizer)}
accuracy = baseline_result['accuracy']
print(f"Baseline accuracy: {accuracy:.4f}")
print(f"Speed: {baseline_result['samples_per_second']:.1f} samples/s")
print(f"Size: {baseline_result['size_MB']:.2f} MB")
model.cpu()
torch.cuda.empty_cache()

GPU: Tesla T4
PyTorch: 2.11.0+cu130 | Transformers: 5.22.1
Baseline accuracy: 0.9084
Speed: 147.6 samples/s
Size: 433.26 MB


In [ ]:
assert 0.9 < accuracy < 0.91

0.9083848627256987

### Task 2: train the model (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://huggingface.co/docs/transformers/en/tasks/sequence_classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

In [9]:
# compared three more QQP checkpoints with the same evaluation function
candidate_names = [
    'textattack/distilbert-base-cased-QQP',
    'textattack/albert-base-v2-QQP',
    'textattack/bert-base-uncased-QQP'
]
results = [baseline_result.copy()]

for candidate_name in candidate_names:
    print('\nEvaluating:', candidate_name, flush=True)
    candidate_tokenizer = transformers.AutoTokenizer.from_pretrained(candidate_name)
    candidate_model = transformers.AutoModelForSequenceClassification.from_pretrained(candidate_name)
    assert candidate_model.config.num_labels == 2

    def tokenize_validation(examples):
        encoded = candidate_tokenizer(
            examples['text1'], examples['text2'],
            padding='max_length', max_length=MAX_LENGTH, truncation=True
        )
        encoded['label'] = examples['label']
        return encoded

    candidate_val = qqp['validation'].map(
        tokenize_validation, batched=True,
        remove_columns=qqp['validation'].column_names
    )
    result = {
        'model': candidate_name,
        **evaluate_model(candidate_model, candidate_val, candidate_tokenizer)
    }
    results.append(result)
    print(result, flush=True)
    del candidate_model, candidate_tokenizer, candidate_val
    gc.collect()
    torch.cuda.empty_cache()


Evaluating: textattack/distilbert-base-cased-QQP


config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  263MB            

... (24 outputs left out)

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

{'model': 'textattack/bert-base-uncased-QQP', 'accuracy': 0.9090774177590898, 'samples_per_second': 142.35039655026216, 'size_MB': 437.943304}


In [10]:
import pandas as pd
from IPython.display import display

comparison = pd.DataFrame(results)
display(comparison.round({'accuracy': 4, 'samples_per_second': 1, 'size_MB': 2}))

best_result = max(results, key=lambda row: row['accuracy'])
fastest_result = max(results, key=lambda row: row['samples_per_second'])
smallest_result = min(results, key=lambda row: row['size_MB'])
best_model_name = best_result['model']
print(f"Highest validation accuracy: {best_model_name} ({best_result['accuracy']:.2%})")
print(f"Fastest: {fastest_result['model']} ({fastest_result['samples_per_second']:.1f} samples/s)")
print(f"Smallest: {smallest_result['model']} ({smallest_result['size_MB']:.2f} MB)")

,model,accuracy,samples_per_second,size_MB
0,gchhablani/bert-base-cased-finetuned-qqp,0.9084,147.6,433.26
1,textattack/distilbert-base-cased-QQP,0.8975,283.4,263.14
2,textattack/albert-base-v2-QQP,0.6316,126.9,46.75
3,textattack/bert-base-uncased-QQP,0.9091,142.4,437.94


Highest validation accuracy: textattack/bert-base-uncased-QQP (90.91%)
Fastest: textattack/distilbert-base-cased-QQP (283.4 samples/s)
Smallest: textattack/albert-base-v2-QQP (46.75 MB)


### report: option B

I compared the original cased BERT with three additional QQP checkpoints: [DistilBERT](https://huggingface.co/textattack/distilbert-base-cased-QQP), [ALBERT](https://huggingface.co/textattack/albert-base-v2-QQP), and [uncased BERT](https://huggingface.co/textattack/bert-base-uncased-QQP). I evaluated all four on the same 40,430 validation pairs in Google Colab on a Tesla T4.

I used each model's own tokenizer, maximum length 128, fixed padding, batch size 32, two data-loader workers, and float32. Models ran in evaluation mode without gradients. Timing started after five warm-up batches, with CUDA synchronization around the measurement. It included data loading, GPU transfers, prediction and accuracy counting, but excluded downloads and tokenization. Size was measured as parameter and buffer storage in decimal MB, not peak GPU memory.

The baseline reached 90.84% accuracy at 147.6 pairs per second and used 433.26 MB. Uncased BERT reached the highest accuracy, 90.91%, at 142.4 pairs per second and used 437.94 MB. Its advantage was only about 0.07 percentage points, so this run did not establish a meaningful general advantage. I selected it for question search using validation accuracy.

DistilBERT reached 89.75% accuracy at 283.4 pairs per second and used 263.14 MB. It was about 1.92 times as fast as the baseline, with accuracy lower by about 1.09 percentage points. This made it a useful speed-and-size alternative when a small accuracy reduction was acceptable.

ALBERT was the smallest model at 46.75 MB, but reached only 63.16% accuracy at 126.9 pairs per second in this run. This unexpectedly low accuracy still needs investigation. I cannot yet tell whether it reflects a checkpoint or preprocessing compatibility issue, so I did not interpret it as evidence that ALBERT is generally worse.

These results compared particular checkpoints with different fine-tuning histories, rather than isolating architecture effects. Speed was measured once per model and may vary between runs. The validation set was also used to select the search model, so these scores were not an independent final test.

### Task 3: try the full pipeline (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

In [11]:
from tqdm.auto import tqdm

# selected the search model using validation accuracy
search_tokenizer = transformers.AutoTokenizer.from_pretrained(best_model_name)
search_model = transformers.AutoModelForSequenceClassification.from_pretrained(best_model_name)
search_model = search_model.to(device).float()
search_model.eval()

# collected unique questions from both training columns
train_questions = list(dict.fromkeys(
    question.strip()
    for column in ('text1', 'text2')
    for question in qqp['train'][column]
    if isinstance(question, str) and question.strip()
))
print('Search model:', best_model_name)
print('Unique training questions:', len(train_questions))

def find_duplicates(question):
    scores = torch.empty(len(train_questions))
    with torch.no_grad():
        for start in tqdm(range(0, len(train_questions), eval_batch_size), desc='Searching'):
            candidates = train_questions[start:start + eval_batch_size]
            inputs = search_tokenizer(
                [question] * len(candidates), candidates,
                padding='max_length', max_length=MAX_LENGTH,
                truncation=True, return_tensors='pt'
            ).to(device)
            logits = search_model(**inputs).logits
            # used QQP label 1 for duplicate pairs
            scores[start:start + len(candidates)] = logits.softmax(dim=1)[:, 1].cpu()

    # excluded the query itself from the results
    for index, candidate in enumerate(train_questions):
        if candidate.casefold() == question.strip().casefold():
            scores[index] = -float('inf')
    top_indices = scores.topk(5).indices.tolist()
    return [(train_questions[index], scores[index].item()) for index in top_indices]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Search model: textattack/bert-base-uncased-QQP
Unique training questions: 493854


### question search and bonus

I kept both implementations. The original `find_duplicates` compared the query with all 493,854 unique training questions using the transformer. I interrupted its first demonstration because the full scan was slow, so I did not obtain a complete query time for this version.

For the bonus, `find_duplicates_fast` first selected up to 200 candidates using TF-IDF cosine similarity, then reranked them with the same transformer and returned the top five. The vocabulary and IDF were fitted only on training questions. Both implementations excluded exact self-matches and used the same model, batch size and maximum token length.

Building the sparse TF-IDF index took 46.0 seconds. The five optimized queries took 2.22, 1.96, 1.82, 2.21 and 2.19 seconds, excluding index construction. All five returned five candidates. Since the full scan was interrupted, I did not measure an end-to-end speedup ratio.

The results for English learning, memory and books were mostly close paraphrases. The Python results also included a narrower question about advanced Python. For the AI versus machine learning query, the first two results were close in meaning, while later results concerned different comparisons, such as AI versus data science. A top-five result was therefore a potential duplicate, not a guaranteed match.

The optimized search was approximate: TF-IDF could miss paraphrases with different wording before the transformer saw them. Increasing the shortlist could improve recall at the cost of more transformer work. I did not measure retrieval recall or agreement with the full scan, so these examples demonstrated functionality rather than proving equal search quality.

In [12]:
example_questions = [
    'How can I learn Python programming?',
    'What are the best ways to learn English?',
    'What is the difference between artificial intelligence and machine learning?',
    'How can I improve my memory?',
    'What are the best books to read?'
]
search_examples = {}
for question in example_questions:
    print('\nQuery:', question, flush=True)
    matches = find_duplicates(question)
    search_examples[question] = matches
    for rank, (candidate, score) in enumerate(matches, start=1):
        print(f'{rank}. {candidate} (duplicate score: {score:.4f})', flush=True)


Query: How can I learn Python programming?


Searching:   0%|          | 0/15433 [00:00<?, ?it/s]

KeyboardInterrupt: 

__Bonus:__ for bonus points, try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

# built a sparse index once using training questions only
index_start = time.perf_counter()
tfidf = TfidfVectorizer(
    ngram_range=(1, 2), max_features=50_000,
    sublinear_tf=True, norm='l2', dtype=np.float32
)
question_vectors = tfidf.fit_transform(train_questions)
print(f'Index built in {time.perf_counter() - index_start:.1f} seconds')
print('Index shape:', question_vectors.shape)

def find_duplicates_fast(question, candidate_count=200):
    if candidate_count < 5:
        raise ValueError('candidate_count must be at least 5')

    # selected candidates by cosine similarity of normalized TF-IDF vectors
    query_vector = tfidf.transform([question])
    lexical_scores = (question_vectors @ query_vector.T).toarray().ravel()
    for index, candidate in enumerate(train_questions):
        if candidate.casefold() == question.strip().casefold():
            lexical_scores[index] = -1

    matching_indices = np.flatnonzero(lexical_scores > 0)
    if len(matching_indices) == 0:
        return []
    count = min(candidate_count, len(matching_indices))
    positions = np.argpartition(
        lexical_scores[matching_indices], len(matching_indices) - count
    )[-count:]
    candidate_indices = matching_indices[positions]
    candidates = [train_questions[index] for index in candidate_indices]

    # reranked only the shortlist with the same transformer
    scores = torch.empty(len(candidates))
    search_model.eval()
    with torch.no_grad():
        for start in range(0, len(candidates), eval_batch_size):
            batch_questions = candidates[start:start + eval_batch_size]
            inputs = search_tokenizer(
                [question] * len(batch_questions), batch_questions,
                padding='max_length', max_length=MAX_LENGTH,
                truncation=True, return_tensors='pt'
            ).to(device)
            logits = search_model(**inputs).logits
            scores[start:start + len(batch_questions)] = logits.softmax(dim=1)[:, 1].cpu()

    top_indices = scores.topk(min(5, len(candidates))).indices.tolist()
    return [(candidates[index], scores[index].item()) for index in top_indices]

Index built in 46.0 seconds
Index shape: (493854, 50000)


In [14]:
example_questions = [
    'How can I learn Python programming?',
    'What are the best ways to learn English?',
    'What is the difference between artificial intelligence and machine learning?',
    'How can I improve my memory?',
    'What are the best books to read?'
]
search_examples = {}
for question in example_questions:
    print('\nQuery:', question, flush=True)
    torch.cuda.synchronize(device)
    query_start = time.perf_counter()
    matches = find_duplicates_fast(question)
    torch.cuda.synchronize(device)
    print(f'Query time: {time.perf_counter() - query_start:.2f} seconds', flush=True)
    if not matches:
        print('No lexical matches found in the training questions.', flush=True)
    search_examples[question] = matches
    for rank, (candidate, score) in enumerate(matches, start=1):
        print(f'{rank}. {candidate} (duplicate score: {score:.4f})', flush=True)


Query: How can I learn Python programming?
Query time: 2.22 seconds
1. How and from where can I learn Python language via video tutorials online? (duplicate score: 0.9935)
2. What are the best sources to learn Python language? (duplicate score: 0.9910)
3. How do I learn Python systematically? (duplicate score: 0.9910)
4. How do I learn Python in easy way? (duplicate score: 0.9876)
5. How can I learn advanced Python? (duplicate score: 0.9846)

Query: What are the best ways to learn English?
Query time: 1.96 seconds
1. What are the best ways to improve English? (duplicate score: 0.9970)
2. What is the best way to learn. English? (duplicate score: 0.9936)
3. What are the best way to learn english.? (duplicate score: 0.9927)
4. How can learn English? (duplicate score: 0.9905)
5. Which is best way to learn English? (duplicate score: 0.9904)

Query: What is the difference between artificial intelligence and machine learning?
Query time: 1.82 seconds
1. What are the main differences between 